# Findrr ML-002: Similarity Evaluator - training and export (v1.1.0)

Implements `docs/ml/models/similarity-evaluator.md`: contrastive fine-tuning of
`sentence-transformers/all-MiniLM-L6-v2` with MultipleNegativesSymmetricRankingLoss,
exported to INT8 ONNX for the Findrr CPU runner.

**Data:** O*NET 31.0 (public, CC BY 4.0). Positives are titles of the same occupation.
Splits keep every occupation in exactly one of train/val/test.

**Evaluation labels are a proxy:** no human-labelled title-similarity benchmark is
available, so graded labels come from O*NET structure (same occupation = 3,
Primary-Short related = 2, Primary-Long related = 1, different major group = 0).
All metrics are computed with the exported INT8 ONNX model, using the same pooling
as the Findrr runner.

**Run on Kaggle:** GPU T4, Internet on, Run All. Extract
`similarity_evaluator_bundle.tar.gz` into `models/similarity-evaluator/v1.1.0/`.

In [ ]:
# Environment report. Reads package metadata only: importing transformers/torch here,
# before the pip install below replaces them, would mix two versions in this kernel.
import platform, sys
from importlib import metadata
print('Python', sys.version.split()[0], '|', platform.platform())
for pkg in ('torch', 'transformers', 'tokenizers', 'datasets', 'sentence-transformers',
            'numpy', 'onnxruntime', 'huggingface-hub'):
    try:
        print(f'{pkg:>22} {metadata.version(pkg)}')
    except metadata.PackageNotFoundError:
        print(f'{pkg:>22} not installed')

In [ ]:
# Pure-Python libraries are pinned exactly; compiled ones (onnx, onnxruntime) get a
# minimum version so pip can choose a prebuilt wheel for Kaggle's Python version.
# The versions actually used are recorded in the manifest metadata.
!pip install -q --prefer-binary sentence-transformers==3.3.1 transformers==4.46.3 \
    datasets==3.1.0 accelerate==1.1.1 "onnx>=1.17.0" "onnxruntime>=1.20.1"

In [ ]:
# O*NET database (CC BY 4.0, U.S. Department of Labor / ETA). Public occupational data only;
# no real user data is used anywhere in this notebook.
ONET_VERSION = '31_0'
ONET_URL = f'https://www.onetcenter.org/dl_files/database/db_{ONET_VERSION}_text.zip'
# SHA-256 of db_31_0_text.zip as downloaded on 2026-11-26. Change both values together.
ONET_SHA256 = '6883548adf5fde64cf6f801b35d15519c9225f2732c3cab0e281c652d16b23a9'
# Optional offline copy: attach the extracted folder as a Kaggle dataset at this path.
ONET_OFFLINE_DIR = '/kaggle/input/onet-db'
SEED = 42

In [ ]:
import hashlib, io, os, zipfile, urllib.request
from pathlib import Path

def obtain_onet_dir() -> Path:
    """Return a directory with the O*NET text files; verify the archive checksum."""
    offline = Path(ONET_OFFLINE_DIR)
    if offline.exists():
        hits = list(offline.rglob('Occupation Data.txt'))
        if hits:
            print(f'Using attached O*NET dataset at {hits[0].parent}')
            return hits[0].parent
    print(f'Downloading {ONET_URL} ...')
    with urllib.request.urlopen(ONET_URL, timeout=120) as resp:
        blob = resp.read()
    digest = hashlib.sha256(blob).hexdigest()
    if digest != ONET_SHA256:
        raise RuntimeError(
            f'O*NET archive checksum mismatch: got {digest}, expected {ONET_SHA256}. '
            'Refusing to train on unverified data.'
        )
    target = Path('/kaggle/working/onet')
    with zipfile.ZipFile(io.BytesIO(blob)) as zf:
        zf.extractall(target)
    return next(target.rglob('Occupation Data.txt')).parent

ONET_DIR = obtain_onet_dir()

In [ ]:
import csv, re, random
from collections import defaultdict
from pathlib import Path

def _find(root: Path, *names: str) -> Path:
    """Return the first existing file among names (O*NET renamed files across releases)."""
    for name in names:
        p = Path(root) / name
        if p.exists():
            return p
    raise FileNotFoundError(f'None of {names} found in {root}')

def _rows(path: Path):
    with open(path, encoding='utf-8', newline='') as fh:
        yield from csv.DictReader(fh, delimiter='\t')

_PAREN = re.compile(r'\s*\([^)]*\)')

def clean_title(title: str, short: str = '') -> str:
    """Prefer the O*NET short title; drop parenthetical expansions; collapse whitespace."""
    base = short if short and short.strip().lower() not in ('n/a', '') else title
    base = _PAREN.sub('', base)
    return ' '.join(base.replace('/', ' / ').split()).strip(' ,;')

def norm(title: str) -> str:
    return ' '.join(re.sub(r'[^\w\s]', ' ', title.lower()).split())

def load_onet(root) -> dict:
    root = Path(root)
    occupations = {r['O*NET-SOC Code']: r['Title'] for r in _rows(_find(root, 'Occupation Data.txt'))}
    reported = defaultdict(list)
    for r in _rows(_find(root, 'Sample of Reported Titles.txt')):
        t = clean_title(r['Reported Job Title'])
        if t and t not in reported[r['O*NET-SOC Code']]:
            reported[r['O*NET-SOC Code']].append(t)
    job_titles = defaultdict(list)
    for r in _rows(_find(root, 'Job Titles.txt', 'Alternate Titles.txt')):
        title_col = 'Job Title' if 'Job Title' in r else 'Alternate Title'
        t = clean_title(r[title_col], r.get('Short Title', ''))
        if t and t not in job_titles[r['O*NET-SOC Code']]:
            job_titles[r['O*NET-SOC Code']].append(t)
    skills = defaultdict(list)
    for r in _rows(_find(root, 'Software Skills.txt', 'Technology Skills.txt')):
        ex = r.get('Workplace Example') or r.get('Example') or ''
        if ex:
            skills[r['O*NET-SOC Code']].append((ex.strip(), r.get('Hot Technology', 'N') == 'Y'))
    related = defaultdict(list)
    for r in _rows(_find(root, 'Related Occupations.txt')):
        related[r['O*NET-SOC Code']].append((r['Related O*NET-SOC Code'], r['Relatedness Tier']))
    return {
        'occupations': occupations,
        'reported': dict(reported),
        'job_titles': dict(job_titles),
        'skills': dict(skills),
        'related': dict(related),
    }

def split_by_occupation(socs, seed: int = 42, fractions=(0.8, 0.1, 0.1)) -> dict:
    """Assign whole occupations to train/val/test, stratified by SOC major group.

    Every title of an occupation lands in exactly one split, so evaluation titles are
    never seen in training (spec: disjoint titles, stratified by occupational domain).
    """
    by_group = defaultdict(list)
    for soc in sorted(set(socs)):
        by_group[soc[:2]].append(soc)
    rng = random.Random(seed)
    assignment = {}
    for group in sorted(by_group):
        members = by_group[group]
        rng.shuffle(members)
        n = len(members)
        n_val = max(1, round(n * fractions[1])) if n >= 3 else 0
        n_test = max(1, round(n * fractions[2])) if n >= 3 else 0
        for i, soc in enumerate(members):
            if i < n_test:
                assignment[soc] = 'test'
            elif i < n_test + n_val:
                assignment[soc] = 'val'
            else:
                assignment[soc] = 'train'
    return assignment

In [ ]:
MAX_TITLES_PER_OCCUPATION = 20
MAX_PAIRS_PER_OCCUPATION = 50
EVAL_PAIRS_PER_TIER = 600
SAMPLES_PER_OCCUPATION = 3
TIER_LABEL = {'same': 3, 'Primary-Short': 2, 'Primary-Long': 1, 'unrelated': 0}

def occupation_titles(onet, soc):
    titles = [onet['occupations'][soc]] + onet['reported'].get(soc, []) + onet['job_titles'].get(soc, [])
    seen, out = set(), []
    for t in titles:
        if norm(t) not in seen:
            seen.add(norm(t)); out.append(t)
    return out[:MAX_TITLES_PER_OCCUPATION]

def build_training_pairs(onet, split_of, seed=42):
    rng = random.Random(seed)
    pairs = []
    for soc in sorted(s for s, sp in split_of.items() if sp == 'train'):
        titles = occupation_titles(onet, soc)
        combos = [(a, b) for i, a in enumerate(titles) for b in titles[i + 1:]]
        rng.shuffle(combos)
        pairs += [{'anchor': a, 'positive': b} for a, b in combos[:MAX_PAIRS_PER_OCCUPATION]]
    rng.shuffle(pairs)
    return pairs

def build_graded_pairs(onet, split_of, split, seed=42):
    """Graded (text_a, text_b, label) pairs; both occupations outside the train split."""
    rng = random.Random(seed)
    socs = sorted(s for s, sp in split_of.items() if sp == split)
    held_out = {s for s, sp in split_of.items() if sp != 'train'}
    tiers = {k: [] for k in TIER_LABEL}
    for soc in socs:
        titles = occupation_titles(onet, soc)
        for _ in range(SAMPLES_PER_OCCUPATION if len(titles) >= 2 else 0):
            a, b = rng.sample(titles, 2)
            tiers['same'].append((a, b, TIER_LABEL['same']))
        for other, tier in onet['related'].get(soc, []):
            if tier in TIER_LABEL and other in held_out and other in onet['occupations']:
                tiers[tier].append((rng.choice(titles), rng.choice(occupation_titles(onet, other)), TIER_LABEL[tier]))
        far = [s for s in socs if s[:2] != soc[:2]]
        for _ in range(SAMPLES_PER_OCCUPATION if far else 0):
            other = rng.choice(far)
            tiers['unrelated'].append((rng.choice(titles), rng.choice(occupation_titles(onet, other)), 0))
    out = []
    for rows in tiers.values():
        rng.shuffle(rows)
        out += rows[:EVAL_PAIRS_PER_TIER]
    return out

def _ranks(values):
    order = sorted(range(len(values)), key=lambda i: values[i])
    ranks = [0.0] * len(values)
    i = 0
    while i < len(order):
        j = i
        while j + 1 < len(order) and values[order[j + 1]] == values[order[i]]:
            j += 1
        for k in range(i, j + 1):
            ranks[order[k]] = (i + j) / 2.0
        i = j + 1
    return ranks

def pearson(x, y):
    n = len(x); mx = sum(x) / n; my = sum(y) / n
    cov = sum((a - mx) * (b - my) for a, b in zip(x, y))
    vx = sum((a - mx) ** 2 for a in x) ** 0.5; vy = sum((b - my) ** 2 for b in y) ** 0.5
    return cov / (vx * vy) if vx and vy else 0.0

def spearman(x, y):
    return pearson(_ranks(list(x)), _ranks(list(y)))

def precision_at(scores, labels, threshold, relevant_min_label=2):
    """Share of pairs scored >= threshold that are relevant (same occupation or Primary-Short)."""
    picked = [l for s, l in zip(scores, labels) if s >= threshold]
    return sum(l >= relevant_min_label for l in picked) / len(picked) if picked else 0.0

In [ ]:
onet = load_onet(ONET_DIR)
split_of = split_by_occupation(onet['occupations'], seed=SEED)
train_pairs = build_training_pairs(onet, split_of, seed=SEED)
val_pairs = build_graded_pairs(onet, split_of, 'val', seed=SEED)
test_pairs = build_graded_pairs(onet, split_of, 'test', seed=SEED + 1)
print(f'train pairs: {len(train_pairs)}, val graded: {len(val_pairs)}, test graded: {len(test_pairs)}')

In [ ]:
import importlib
def _check(module, expected):
    found = importlib.import_module(module).__version__
    if found != expected:
        raise RuntimeError(
            f'{module} {found} is loaded but {expected} is pinned. The kernel imported it '
            'before the pip install finished. Use Run > Restart & clear cell outputs, then '
            'Run All again.')
_check('transformers', '4.46.3')
_check('datasets', '3.1.0')
_check('sentence_transformers', '3.3.1')

In [ ]:
import numpy as np, torch
from datasets import Dataset
from sentence_transformers import (SentenceTransformer, SentenceTransformerTrainer,
                                   SentenceTransformerTrainingArguments, losses)
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator
from sentence_transformers.training_args import BatchSamplers
from transformers import set_seed
set_seed(SEED)
BASE_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'
model = SentenceTransformer(BASE_MODEL)
model.max_seq_length = 128
loss = losses.MultipleNegativesSymmetricRankingLoss(model)
val_eval = EmbeddingSimilarityEvaluator([a for a, _, _ in val_pairs], [b for _, b, _ in val_pairs],
                                        [l / 3.0 for _, _, l in val_pairs], name='onet-val')
args = SentenceTransformerTrainingArguments(
    output_dir='/kaggle/working/ml002-checkpoints', num_train_epochs=4,
    per_device_train_batch_size=64, learning_rate=2e-5, warmup_ratio=0.1,
    fp16=torch.cuda.is_available(), batch_sampler=BatchSamplers.NO_DUPLICATES,
    eval_strategy='epoch', save_strategy='epoch', save_total_limit=1,
    load_best_model_at_end=True, metric_for_best_model='eval_onet-val_spearman_cosine',
    seed=SEED, report_to='none',
)
trainer = SentenceTransformerTrainer(model=model, args=args, train_dataset=Dataset.from_list(train_pairs),
                                     loss=loss, evaluator=val_eval)
trainer.train()

## Export to ONNX, quantise to INT8, then evaluate the shipped model

In [ ]:
import shutil
import onnxruntime as ort
from onnxruntime.quantization import QuantType, quantize_dynamic
EXPORT_DIR = Path('/kaggle/working/similarity-evaluator-v1.1.0')
shutil.rmtree(EXPORT_DIR, ignore_errors=True); EXPORT_DIR.mkdir(parents=True)
hf_model = model[0].auto_model.cpu().eval()
tok = model.tokenizer
dummy = tok(['Software Engineer', 'Registered Nurse'], return_tensors='pt', padding=True)
fp32_path = Path('/kaggle/working/ml002-fp32.onnx')
torch.onnx.export(
    hf_model, (dummy['input_ids'], dummy['attention_mask'], dummy['token_type_ids']), str(fp32_path),
    input_names=['input_ids', 'attention_mask', 'token_type_ids'], output_names=['last_hidden_state'],
    dynamic_axes={n: {0: 'batch', 1: 'seq'} for n in ['input_ids', 'attention_mask', 'token_type_ids', 'last_hidden_state']},
    opset_version=17,
)
quantize_dynamic(str(fp32_path), str(EXPORT_DIR / 'model.onnx'), weight_type=QuantType.QInt8)
tok.save_pretrained(EXPORT_DIR)
(EXPORT_DIR / 'config.json').write_text(json.dumps({
    'embedding_dimension': 384, 'max_seq_length': 128, 'pooling': 'mean', 'normalize': True}, indent=2))
(EXPORT_DIR / 'preprocess_config.json').write_text(json.dumps({'lowercase': True, 'strip_accents': True}, indent=2))

session = ort.InferenceSession(str(EXPORT_DIR / 'model.onnx'), providers=['CPUExecutionProvider'])

def onnx_embed(text):
    """Identical to findrr SimilarityEvaluatorRunner._embed_text."""
    enc = tok(text, truncation=True, max_length=128, return_tensors='np')
    feeds = {'input_ids': enc['input_ids'], 'attention_mask': enc['attention_mask']}
    if 'token_type_ids' in [i.name for i in session.get_inputs()]:
        feeds['token_type_ids'] = enc.get('token_type_ids', np.zeros_like(enc['input_ids']))
    tokens = session.run(None, feeds)[0]
    mask = np.expand_dims(enc['attention_mask'], -1)
    emb = (tokens * mask).sum(1) / np.clip(mask.sum(1), 1e-9, None)
    emb = emb / np.clip(np.linalg.norm(emb, axis=1, keepdims=True), 1e-9, None)
    return emb[0]

def onnx_sim(a, b):
    return float(np.clip(np.dot(onnx_embed(a), onnx_embed(b)), 0.0, 1.0))

In [ ]:
scores = [onnx_sim(a, b) for a, b, _ in test_pairs]
labels = [l for _, _, l in test_pairs]
st = model.to('cpu')
parity = float(np.mean([np.dot(onnx_embed(t), st.encode(t, normalize_embeddings=True))
                        for t, _, _ in test_pairs[:200]]))
measured = {
    'pearson_r': pearson(scores, labels),
    'spearman_rho': spearman(scores, labels),
    'precision_at_0_70': precision_at(scores, labels, 0.70),
    'ac_ml_002_1_synonym_similarity': onnx_sim('Software Developer', 'Software Engineer'),
    'ac_ml_002_2_distinct_similarity': onnx_sim('Software Developer', 'Dental Assistant'),
    'onnx_int8_vs_pytorch_cosine': parity,
    'test_pairs': float(len(test_pairs)),
}
print(json.dumps(measured, indent=2))

In [ ]:
import hashlib, json
from datetime import datetime, timezone
from pathlib import Path

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, 'rb') as fh:
        while chunk := fh.read(65536):
            h.update(chunk)
    return h.hexdigest()

def build_manifest(export_dir, model_id, model_name, version, architecture,
                   evaluation_summary, acceptance, metadata) -> dict:
    """Manifest in the exact schema of findrr.ml.manifest.ModelManifest.

    ``files`` maps every exported file (except manifest.json) to its SHA-256.
    ``evaluation_summary`` holds measured floats only; nothing is defaulted.
    """
    export_dir = Path(export_dir)
    files = {
        p.name: sha256_file(p)
        for p in sorted(export_dir.iterdir())
        if p.is_file() and p.name != 'manifest.json'
    }
    for key, value in evaluation_summary.items():
        if not isinstance(value, (int, float)) or value != value:  # reject None/NaN
            raise ValueError(f'Metric {key!r} was not measured: {value!r}')
    return {
        'model_id': model_id,
        'model_name': model_name,
        'version': version,
        'architecture': architecture,
        'export_date': datetime.now(timezone.utc).isoformat(),
        'files': files,
        'evaluation_summary': {k: float(v) for k, v in evaluation_summary.items()},
        'metadata': {**metadata, 'acceptance': acceptance},
    }

def check_acceptance(measured: dict, criteria: dict) -> dict:
    """criteria: {metric: ('>=' | '<', threshold)} taken verbatim from the model spec."""
    results = {}
    for metric, (op, threshold) in criteria.items():
        value = measured[metric]
        ok = value >= threshold if op == '>=' else value < threshold
        results[metric] = {'value': float(value), 'op': op, 'threshold': threshold, 'passed': bool(ok)}
    return {'passed': all(r['passed'] for r in results.values()), 'criteria': results}

In [ ]:
# Thresholds copied verbatim from docs/ml/models/similarity-evaluator.md (approved).
ACCEPTANCE = check_acceptance(measured, {
    'pearson_r': ('>=', 0.85),
    'precision_at_0_70': ('>=', 0.90),
    'ac_ml_002_1_synonym_similarity': ('>=', 0.85),
    'ac_ml_002_2_distinct_similarity': ('<', 0.30),
})

In [ ]:
# Acceptance gate. A model that fails the approved criteria is NOT exported unless you
# deliberately set ALLOW_EXPORT_BELOW_ACCEPTANCE = True. In that case the manifest records
# acceptance.passed = false and the Findrr app refuses to load the model.
ALLOW_EXPORT_BELOW_ACCEPTANCE = False
print(json.dumps(ACCEPTANCE, indent=2))
if not ACCEPTANCE['passed'] and not ALLOW_EXPORT_BELOW_ACCEPTANCE:
    raise RuntimeError(
        'ML-002 did not meet its acceptance criteria. Not exporting. '
        'Report the metrics above; do not change thresholds without approval.'
    )

In [ ]:
import tarfile, sentence_transformers, transformers
manifest = build_manifest(
    EXPORT_DIR, 'ML-002', 'similarity-evaluator', '1.1.0',
    'sentence-transformers/all-MiniLM-L6-v2 (ONNX INT8, mean pooling)',
    evaluation_summary=measured, acceptance=ACCEPTANCE,
    metadata={'dataset': f'O*NET {ONET_VERSION} (CC BY 4.0)', 'dataset_sha256': ONET_SHA256,
              'evaluation_labels': 'proxy graded labels from O*NET structure (0-3)',
              'split': 'by occupation, stratified by SOC major group (80/10/10)',
              'train_pairs': len(train_pairs), 'seed': SEED,
              'sentence_transformers': sentence_transformers.__version__,
              'transformers': transformers.__version__, 'onnxruntime': ort.__version__},
)
(EXPORT_DIR / 'manifest.json').write_text(json.dumps(manifest, indent=2))
BUNDLE = '/kaggle/working/similarity_evaluator_bundle.tar.gz'
with tarfile.open(BUNDLE, 'w:gz') as tar:
    for p in sorted(EXPORT_DIR.iterdir()):
        tar.add(p, arcname=p.name)
print(json.dumps(manifest, indent=2))

In [ ]:
# Re-verify the bundle exactly as the Findrr app will (SHA-256 of every file).
import tarfile, tempfile
with tempfile.TemporaryDirectory() as tmp:
    with tarfile.open('/kaggle/working/similarity_evaluator_bundle.tar.gz') as tar:
        tar.extractall(tmp)
    m = json.loads((Path(tmp) / 'manifest.json').read_text())
    for name, digest in m['files'].items():
        assert sha256_file(Path(tmp) / name) == digest, name
print('[OK] Bundle verified:', len(m['files']), 'files;', 'acceptance passed =', m['metadata']['acceptance']['passed'])